# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SupreetOP/Flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Search performance can identify content opportunities

The paper reports findings that connect observed search-performance signals with opportunities to improve content performance.

**Methodology question:** Where exactly does the label or outcome used to define an opportunity come from? I would want to verify whether it is based on an independently observed future outcome, a manually defined rule, or another derived signal. If the label is derived from the same search-performance information used as an input, there could be a risk of circularity or leakage.

I would also check whether the validation design keeps the information used to create the prediction separate from the period used to measure the outcome. A time-aware design would make the direction of prediction clearer.

### Finding 2 — Content changes can improve search performance

The paper reports evidence that content changes are associated with improved search performance.

**Methodology question:** Does the validation design support interpreting the observed improvement as an effect of the content change, or does it only show that performance changed after the change? I would want to understand how the study handles other factors such as seasonality, changing search demand, algorithm changes, or differences between the pages that were changed and those that were not.

A useful validation design would make the comparison group and observation window explicit and ensure that future performance information is not available when the intervention or prediction is defined.

### Overall audit perspective

These questions are not a rejection of the findings. They are checks on whether the label construction and validation design support the strength of the conclusions being made. The same standard should be applied to my own Week-5 model.

## 2. My model under an honest split (before/after)

The Week-5 model used a random validation split. For a more realistic evaluation, I re-ran the model using a time-aware split: March 2026 features were used to predict April 2026 outcomes.

The comparison below shows the Week-5 random-split result against the time-aware result. This checks whether the model's performance holds when the validation setup better matches the real decision process.

## 3. Leakage audit

I audited the final feature set for three types of leakage: label-derived features, future-window features, and product or decision flags.

The final model uses only information available at the decision moment:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

The future April GSC clicks are used only as the outcome for validation and are not included as model features.

The audit checks the final feature list for suspicious future, label-derived, or product-flag fields. A clean result supports the conclusion that the model inputs do not directly contain the future outcome.

In [1]:
!pip -q install duckdb huggingface_hub pandas scikit-learn

In [9]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET hf_secret ("
    f"TYPE HUGGINGFACE, "
    f"TOKEN '{HF_TOKEN}'"
    f")"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [10]:
final_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

print("FINAL MODEL FEATURES")
print("=" * 60)

for feature in final_features:
    print("-", feature)

FINAL MODEL FEATURES
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions


In [11]:
suspicious_keywords = [
    "future",
    "outcome",
    "label",
    "target",
    "next_month",
    "product_flag",
    "action",
    "refresh",
    "quick_win",
    "stale"
]

suspicious_features = [
    feature for feature in final_features
    if any(keyword in feature.lower() for keyword in suspicious_keywords)
]

print("=" * 60)
print("LEAKAGE FEATURE CHECK")
print("=" * 60)

print("Suspicious features found:", suspicious_features)

if len(suspicious_features) == 0:
    print("RESULT: PASS")
    print("No obvious future, target, product-flag, or label-derived features found.")
else:
    print("RESULT: REVIEW REQUIRED")

LEAKAGE FEATURE CHECK
Suspicious features found: []
RESULT: PASS
No obvious future, target, product-flag, or label-derived features found.


In [12]:
target_column = "future_gsc_clicks"

all_model_columns = set(final_features)

leakage_overlap = all_model_columns.intersection({target_column})

print("=" * 60)
print("FEATURE vs TARGET LEAKAGE CHECK")
print("=" * 60)

print("Target column:", target_column)
print("Features used:", final_features)
print("Overlap:", leakage_overlap)

if len(leakage_overlap) == 0:
    print("\nRESULT: PASS")
    print("The future target is not included in the model feature set.")
else:
    print("\nRESULT: FAIL")
    print("The target appears in the model features.")

FEATURE vs TARGET LEAKAGE CHECK
Target column: future_gsc_clicks
Features used: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']
Overlap: set()

RESULT: PASS
The future target is not included in the model feature set.


In [13]:
future_columns = [
    col for col in final_features
    if any(word in col.lower() for word in [
        "future",
        "next",
        "forecast"
    ])
]

print("=" * 60)
print("FUTURE-WINDOW FEATURE CHECK")
print("=" * 60)

print("Future-window features:", future_columns)

if len(future_columns) == 0:
    print("\nRESULT: PASS")
    print("No future-window features are included in the model.")
else:
    print("\nRESULT: FAIL")
    print("Future-window features were found.")


FUTURE-WINDOW FEATURE CHECK
Future-window features: []

RESULT: PASS
No future-window features are included in the model.


### Leakage audit conclusion

The final feature set contains only decision-time performance signals: GSC impressions, GSC clicks, GSC average position, GA4 pageviews, and GA4 engaged sessions.

The audit found no suspicious product flags, target-derived features, or future-window features. The future April GSC clicks target was also confirmed to be absent from the model feature set.

Based on these checks, the feature set passes the leakage audit. This supports using the model as directional decision-support, although the audit does not prove that every possible source of leakage has been eliminated.

## 4. Claim rewrite

### Original claim

"The Random Forest model provides better content-opportunity predictions than the Week-4 baseline."

### Safer claim

"On the measured validation data, the Random Forest achieved higher ROC-AUC and PR-AUC than the Week-4 rule-based baseline. The observed improvement is directional evidence that the learned model captures useful patterns in the available decision-time features. However, this result should be treated as decision-support rather than proof that the model will consistently outperform the baseline on future data."

### Why the wording is safer

The result was measured on a specific validation setup and therefore should not be generalized beyond the tested data. The model's false positives and false negatives also show that the available features do not capture every factor affecting future content performance.

## Self-check

- [x] Every section above is filled with the required markdown explanations and supporting code/results.
- [x] The notebook runs from top to bottom without errors using Runtime → Run all.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] The Week-5 model was re-evaluated under an honest validation split and the before/after results were reviewed.
- [x] The final feature set was audited for target-derived, future-window, and suspicious product/decision features.
- [x] Model limitations and the strength of the claims were addressed rather than overstated.
- [x] The notebook has been committed and pushed to my repository under `work/notebooks/w06_validation_audit.ipynb`, and the repository URL has been submitted on the assignment card.